# Exp 031 — CoT user-state prompt + Qwen 2.5-3B (DEVSET)

**Three-way pair-test sibling of 029 / 030.** Same retrieval (wRRF), same CoT prompt (`response_generation_cot_user_state.txt`), same `max_new_tokens=192`. Only `lm_type` changes:

- 029: Qwen 2.5-**1.5B** (champion model, ~67% format compliance in local smoke)
- **031: Qwen 2.5-3B** (this notebook — practical sweet spot, expected ~85%)
- 030: Qwen 2.5-**7B** (slow at batch 4 due to OOM)

**Why 3B is the practical sweet spot**: large enough to follow the structured `<user_state>` block reliably (the confound that hit 1.5B), small enough to run 8000 rows in ~12-18 min at batch 16 (vs 100+ min for 7B at batch 4). And the CoT prompt's word-bans target exactly the AI-speak failure mode that hit 3B + stock prompt on Blind-A (exp 024).

**Risk being tested**: exp 024 shipped Qwen 3B + stock prompt + max_new_tokens=192 to Blind-A and it regressed (composite 0.33→0.29, LLM 3.15→3.00). The new CoT prompt explicitly bans 'fantastic', 'perfectly captures', etc. If 3B+CoT cleanly produces grounded responses on dev, the 3B path opens; if responses still drift to filler, the bigger-model penalty is structural and we revert to 1.5B.

Wall time on A100: ~12-18 min for 8000 rows at batch 16.

In [ ]:
# 1) Verify GPU. 3B fits T4/L4/A100; A100 strongly recommended for batch 16 speed.
!nvidia-smi | head -20

In [ ]:
# 2) FORCE-FRESH clone — pull latest fresh-model code.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()
!echo -n 'branch:  ' && git rev-parse --abbrev-ref HEAD

In [ ]:
# 3) Install deps.
!pip install -q -r requirements.txt
!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 4) Experiment parameters.
TID = '031-cot-user-state-qwen3b-devset'
# 3B at bf16+sdpa: batch 16 fits A100 40GB with headroom (3B weights
# ~6 GB). Bump to 32 if you confirm via nvidia-smi mid-run that you
# have ~10 GB headroom. Drop to 8 on T4 16GB.
BATCH_SIZE = 16
ATTN = 'sdpa'
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Run devset two-step inference.
# response_max_new_tokens=192 (set in yaml) to fit user_state block + response.
!cd music-crs-baselines && PYTORCH_ALLOC_CONF=expandable_segments:True \
    python run_inference_devset.py \
    --tid {TID} \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 6) Validate prediction JSON + zip for download.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/devset/{TID}.json'
assert os.path.isfile(SRC), f'prediction not found at {SRC} — did inference fail?'

with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)} (expected 8000)')
assert len(rows) >= 8000, f'only {len(rows)} rows — partial run; do not score'

stage = f'/content/_stage_{TID}'
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, f'{TID}.json'))
zip_base = f'/content/{TID}'
shutil.make_archive(zip_base, 'zip', stage)
print('wrote', zip_base + '.zip')
!ls -lh {zip_base}.zip

In [ ]:
# 7a) Browser download.
from google.colab import files
files.download(f'/content/{TID}.zip')

In [ ]:
# 7b) Drive backup.
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
dst_dir = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(f'/content/{TID}.zip', dst_dir)
shutil.copy(f'music-crs-baselines/exp/inference/devset/{TID}.json', dst_dir)
print(f'saved to Drive: {dst_dir}')
!ls -lh {dst_dir}

In [ ]:
# 8) Quality probe — sample responses + parser-leak rate.
# The CoT post-processor in crs_baseline.extract_cot_response prefers
# <response>...</response>; if missing, it strips <user_state>...</user_state>
# and returns the rest. The FINAL parsed text going to Gemini should be
# near-zero leak (no <user_state> / <response> tags, no field names like 'mood:').
import json, random, re

with open(f'music-crs-baselines/exp/inference/devset/{TID}.json') as f:
    rows = json.load(f)

field_leak_re = re.compile(
    r'^(?:mood|intent|energy|sonic_pref|era_pref|familiarity):',
    re.M,
)
tag_leak_re = re.compile(r'<\s*/?\s*(user_state|response)\s*>', re.I)

leak_field, leak_tag, empty = 0, 0, 0
for r in rows:
    resp = (r.get('predicted_response') or '').strip()
    if not resp:
        empty += 1
        continue
    if field_leak_re.search(resp):
        leak_field += 1
    if tag_leak_re.search(resp):
        leak_tag += 1

n = len(rows)
print(f'rows total            : {n}')
print(f'empty responses       : {empty}  ({empty/n:.1%})')
print(f'field-name leak       : {leak_field}  ({leak_field/n:.1%})')
print(f'tag leak              : {leak_tag}  ({leak_tag/n:.1%})')
print()
print('=== 10 random sample responses ===')
random.seed(42)
for i in random.sample(range(n), min(10, n)):
    print(f'\n[{i}] turn {rows[i].get("turn_number")}')
    print(rows[i].get('predicted_response', '')[:400])

# Heuristic gate: tag/field-leak rate >5% in the FINAL parsed response
# means parser failure (unexpected — local smoke had 0% leak). It is fine
# if the LM only emits the structured <user_state> block ~70% of the time
# (1.5B drops the format on some queries — local smoke showed ~67%
# follow-rate). The parser falls back cleanly when the format is missing.

## After the Colab run, on local M4:

```bash
cd recsys2026
TID=031-cot-user-state-qwen3b-devset
unzip -o ~/Downloads/${TID}.zip -d music-crs-baselines/exp/inference/devset/
source recsys26/bin/activate
python scripts/local_eval.py --tid ${TID} --split dev
pytest tests/test_wave2_integration.py -v
```

## Decision gate (compared to 029 / 1.5B-CoT and exp 024 / 3B-stock-prompt)

- **Response quality clearly more grounded than 029 AND no AI-speak filler ('fantastic', 'perfectly')** → 3B+CoT is the new candidate; promote to a Blind-A run (still subject to fresh-model gate).
- **Format-compliance ≥85% AND comparable response specificity to 029** → CoT is the dominant lever; 1.5B cheaper and equally good for production.
- **Filler words leak through despite the prompt's bans** → 3B's stock-prompt failure mode (exp 024) is structural; revert to 1.5B.